# Lab Assignment: Generative AI

**Lab team:** 16

**Name (member 1):** Eva Blázquez Pardo

**Name (member 2):** Gabriela Damas García

# Exercise 1:  Generative AI based on diffusion models: Brownian motion

In this exercise the training data is generated by injecting noise using Brownian motion (variance exploding diffusion model).
The stochastic differential equation (SDE) that characterizes this forward diffusion process is
$$
d\mathbf{x}(t) = \sigma^t d\mathbf{W}(t)
$$
  
with $\mathbf{x}(0) \sim p_0 \left(\mathbf{x} \right)$, and  $\sigma > 0 $.

1. What is the distribution of $\mathbf{x}(t)$ assuming that $\mathbf{x}(0) = \mathbf{x}_0$ for arbitrary 0

    Integrating the stochastic differential equation between $0$ and $t$, we obtain

    $$
    x(t) = x_0 + \int_0^t g(s)\, dW(s).
    $$

    Since $x_0$ is constant and the Itô integral of a deterministic function with respect to Brownian motion is a Gaussian variable, it follows that $x(t)$ is a normally distributed random variable.

    Its conditional mean is

    $$
    \mathbb{E}[x(t) \mid x(0) = x_0] = x_0.
    $$

    Its conditional covariance is

    $$
    \mathrm{Cov}[x(t) \mid x(0) = x_0]
    =
    \left( \int_0^t g^2(s)\, ds \right) I.
    $$

    In this case, the diffusion coefficient is $g(s) = \sigma^s$, so

    $$
    \mathrm{Cov}[x(t) \mid x(0) = x_0]
    =
    \left( \int_0^t \sigma^{2s}\, ds \right) I.
    $$

    Now we compute the integral:

    $$
    \int_0^t \sigma^{2s}\, ds
    =
    \int_0^t e^{2s \ln \sigma}\, ds
    =
    \frac{e^{2t \ln \sigma} - 1}{2 \ln \sigma}
    =
    \frac{\sigma^{2t} - 1}{2 \ln \sigma}.
    $$

    Therefore, the distribution of $x(t)$ conditioned on $x_0$ is

    $$
    x(t) \mid x(0) = x_0 \sim \mathcal{N}\left(
    x_0,\;
    \frac{\sigma^{2t} - 1}{2 \ln \sigma}\, I
    \right).
    $$

2. What is the SDE for the time-reversed process?

    In particular, the reverse-time SDE is given by

    $$
    d\mathbf{x}(t) = - \sigma^{2t} \, \nabla_{\mathbf{x}} \log p_t(\mathbf{x}(t)) \, dt + \sigma^t \, d\bar{\mathbf{W}}(t).
    $$

    Here, $p_t(\mathbf{x})$ denotes the probability density of $\mathbf{x}(t)$, and $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$ is the *score* function.

    The first term, $- \sigma^{2t} \, \nabla_{\mathbf{x}} \log p_t(\mathbf{x}(t))$, acts as a drift term that pushes samples toward high-probability regions, while the second term, $\sigma^t \, d\bar{\mathbf{W}}(t)$, represents stochastic noise.

    Therefore, unlike the forward process, which only adds noise, the reverse process not only removes noise but also guides samples toward the data distribution.

3. For the synthesis of new images

    1. In what interval is the reverse process SDE integrated?

        For the generation of new images, the reverse-time SDE is integrated backward in time.

        In particular, integration is performed over the interval

        $$
        t \in [T, 0],
        $$

        starting from a large time $T$ (where data are close to pure noise) and progressing to $t = 0$, where samples follow the data distribution.

    2. From which distribution $\pi\left(\mathbf{x} \right)$ is the initial condition of the reverse SDE sampled?

        The initial condition of the reverse SDE is obtained by sampling from the forward-process distribution at time $T$.

        For a sufficiently large value of $T$, the distribution of $\mathbf{x}(T)$ is approximately Gaussian. In this case, using the result obtained in section 1, the initial condition is taken from

        $$
        \pi(\mathbf{x}) =
        \mathcal{N}\left(
        0,\;
        \frac{\sigma^{2T} - 1}{2 \ln \sigma} \, \mathbf{I}
        \right).
        $$

        That is, a Gaussian distribution with zero mean and covariance proportional to the identity matrix.

## Exercise 1.1:  Training the model by weighted sum of denoising score matching objectives

1. Give the mathematical form and explain the cost function used in the training of the generative AI model.

    Training of the generative model is based on a weighted \textit{denoising score matching} objective.

    For a Gaussian diffusion process, the forward perturbation model is given by

    $$
    \mathbf{x}(t) \mid \mathbf{x}_0 = \mu_{\mathbf{x}_0}(t) + \sigma(t)\mathbf{Z},
    \quad \mathbf{Z} \sim \mathcal{N}(0, \mathbf{I}),
    $$

    so that

    $$
    p_{t|0}(\mathbf{x}(t), t \mid \mathbf{x}_0, 0)
    =
    \frac{1}{(2\pi)^{D/2}\sigma^D(t)}
    \exp\left(
    -\frac{1}{2\sigma^2(t)} \|\mathbf{x}(t) - \mu_{\mathbf{x}_0}(t)\|_2^2
    \right).
    $$

    From this expression, the score function is

    $$
    \nabla_{\mathbf{x}} \log p_{t|0}(\mathbf{x}(t), t \mid \mathbf{x}_0, 0)
    =
    -\frac{1}{\sigma^2(t)} \left(\mathbf{x}(t) - \mu_{\mathbf{x}_0}(t)\right)
    =
    -\frac{1}{\sigma(t)} \mathbf{Z}.
    $$

    The loss function is then defined as

    $$
    \mathrm{loss}(\theta)
    =
    \mathbb{E}_{t \sim U[0,1]}
    \left[
    \lambda(t)
    \mathbb{E}_{\mathbf{x}_0 \sim p_0(\mathbf{x})}
    \left[
    \mathbb{E}_{\mathbf{x} \sim p_{t|0}(\mathbf{x}, t \mid \mathbf{x}_0, 0)}
    \left[
    \left\|
    s(\mathbf{x}, t; \theta)
    -
    \nabla_{\mathbf{x}} \log p_{t|0}(\mathbf{x}, t \mid \mathbf{x}_0, 0)
    \right\|_2^2
    \right]
    \right]
    \right].
    $$

    Substituting the score expression, we obtain

    $$
    \mathrm{loss}(\theta)
    =
    \mathbb{E}_{t \sim U[0,1]}
    \left[
    \mathbb{E}_{\mathbf{x}_0 \sim p_0(\mathbf{x})}
    \left[
    \mathbb{E}_{\mathbf{x} \sim p_{t|0}(\mathbf{x}, t \mid \mathbf{x}_0, 0)}
    \left[
    \frac{\lambda(t)}{\sigma^2(t)}
    \left\|
    \sigma(t)s(\mathbf{x}, t; \theta) + \mathbf{Z}
    \right\|_2^2
    \right]
    \right]
    \right].
    $$

    Next, using an empirical estimate with the training samples $\{\mathbf{x}_i\}_{i=1}^N$, it is approximated by

    $$
    \mathrm{loss}(\theta)
    \approx
    \frac{1}{N}
    \sum_{i=1}^N
    \frac{\lambda(t_i)}{\sigma^2(t_i)}
    \left\|
    \sigma(t_i)s(\mathbf{x}_i(t_i), t_i; \theta) + \mathbf{Z}_i
    \right\|_2^2,
    $$

    where

    $$
    t_i \sim U[0,1],
    \quad \mathbf{Z}_i \sim \mathcal{N}(0, \mathbf{I}),
    \quad \mathbf{x}_i(t_i) = \mu_{\mathbf{x}_i}(t_i) + \sigma(t_i)\mathbf{Z}_i.
    $$

    Finally, by choosing

    $$
    \lambda(t) \propto \sigma^2(t),
    $$

    the loss function simplifies to

    $$
    \mathrm{loss}(\theta)
    \propto
    \frac{1}{N}
    \sum_{i=1}^N
    \left\|
    \sigma(t_i)s(\mathbf{x}_i(t_i), t_i; \theta) + \mathbf{Z}_i
    \right\|_2^2.
    $$

2. Indicate what type of neural network is used to model the score, its inputs, outputs, and architecture.

    The score is modeled using a deep neural network based on a U-Net architecture.

    The network takes as input a noisy sample $\mathbf{x}(t)$ and the corresponding time variable $t$, which represents the noise level.

    The time variable is embedded using Gaussian random Fourier features, allowing the model to incorporate temporal information into the network.

    The output of the network is

    $$
    s_\theta(\mathbf{x}(t), t),
    $$

    which is an approximation of the score function

    $$
    \nabla_{\mathbf{x}} \log p_t(\mathbf{x}),
    $$

    and has the same dimensionality as the input image.

    The architecture follows a U-Net structure, consisting of an encoder-decoder design:

    - The encoder progressively reduces the spatial resolution using convolutional layers while extracting high-level features.  
    - The decoder reconstructs the spatial resolution using transposed convolutions.  
    - Skip connections are used to preserve fine-grained information by connecting corresponding layers in the encoder and decoder.  

    This architecture is well suited for image data, as it captures both global and local structures, which is essential for accurately estimating the score at different noise levels.

3. Explain how time is input in the neural network model for the time-dependent score function using random Fourier features.

    The time variable $t$ is incorporated into the neural network using Gaussian random Fourier features.

    Instead of feeding the scalar time variable directly into the network, it is first transformed into a higher-dimensional feature representation using sinusoidal functions. In particular, the embedding is constructed as

    $$
    (\sin(\omega_i t), \cos(\omega_i t)),
    $$

    where the frequencies $\omega_i$ are randomly sampled from a Gaussian distribution at initialization and remain fixed during training.

    This transformation maps the scalar time variable into a vector that captures information at multiple frequencies, allowing the model to better represent the dependence on $t$.

    The resulting embedding is then passed through a linear layer and injected into the network at different stages, typically by adding it to the feature maps of convolutional layers.

    This approach enables the neural network to condition its predictions on the noise level, which is essential for learning a time-dependent score function and accurately modeling the reverse diffusion process.

4. Illustrate how to train a model to generate handwritten digits for the MNIST dataset using the Brownian motion diffusion model.

    This first block of code should run in less than 5 minutes.

    See the python file [brownian](src/brownian.py) and notebook [train_mnist_bm.ipynb](train/train_mnist_bm.ipynb) for the full implementation.

## Exercise 1.2: Generation using the Euler-Maruyama integrator

Use a model that you have previously trained (not the one in the previous exercise) high-quality model to generate some samples using the Euler-Maruyama integrator.

The implementation details and experimental results can be found in the corresponding notebook [sampler_mnist_bm.ipynb](samplers/sampler_mnist_bm.ipynb)

## Exercise 1.3: Generation using a probability flow ODE.

1. Explain what is the Fokker-Planck equation and in what way is it related to an SDE.

    The Fokker-Planck equation is a partial differential equation that describes the time evolution of the probability density associated with a stochastic process. In other words, while a stochastic differential equation (SDE) describes the dynamics of the random variable itself, the Fokker-Planck equation describes the dynamics of its probability distribution.

    Let us consider the Itô stochastic differential equation

    $$
    dX(t)=f(X(t),t)\,dt+\sqrt{2D(X(t),t)}\,dW(t),
    $$

    where $f(X(t),t)$ is the drift term, $D(X(t),t)$ is the diffusion coefficient, and $W(t)$ is a Wiener process (standard Brownian motion).

    This SDE gives a \emph{trajectory-wise} description of the process: it specifies how each random realization $X(t)$ evolves over time. However, in many applications we are not only interested in individual trajectories, but in the probability density of the state at time $t$.

    If the initial condition is fixed as

    $$
    X(t_0)=x_0,
    $$

    then the relevant density is the conditional density

    $$
    p(x,t\mid x_0,t_0).
    $$

    The corresponding Fokker-Planck equation is

    $$
    \partial_t p(x,t\mid x_0,t_0)
    =
    -\partial_x\!\left[f(x,t)\,p(x,t\mid x_0,t_0)\right]
    +
    \partial_x^2\!\left[D(x,t)\,p(x,t\mid x_0,t_0)\right].
    $$

    This equation describes how the conditional probability density evolves in time under the action of two effects:

    1. The term
    $$
    -\partial_x\!\left[f(x,t)\,p(x,t\mid x_0,t_0)\right]
    $$
    represents the deterministic transport of probability mass due to the drift $f(x,t)$.

    2. The term
    $$
    \partial_x^2\!\left[D(x,t)\,p(x,t\mid x_0,t_0)\right]
    $$
    represents the spreading of probability mass caused by the stochastic diffusion.

    Therefore, the SDE and the Fokker-Planck equation are two equivalent ways of describing the same stochastic dynamics:

    - the SDE acts at the level of random sample paths;
    - the Fokker-Planck equation acts at the level of probability densities.

    If, instead of fixing the initial condition at a single point $x_0$, we assume that the initial state is distributed according to some density $p_0(x)$ at time $t_0$, then the full density at time $t$ is obtained by marginalizing the conditional density:

    $$
    p(x,t)=\int p(x,t\mid x_0,t_0)\,p_0(x_0)\,dx_0.
    $$

    In this case, the corresponding Fokker-Planck equation for the general probability density is

    $$
    \partial_t p(x,t)
    =
    -\partial_x\!\left[f(x,t)\,p(x,t)\right]
    +
    \partial_x^2\!\left[D(x,t)\,p(x,t)\right],
    $$

    with initial condition

    $$
    p(x,t)\big|_{t=t_0}=p_0(x).
    $$

    Hence, the relationship between the Fokker--Planck equation and an SDE can be summarized as follows:

    - the SDE specifies the stochastic evolution of the variable $X(t)$;
    - the Fokker-Planck equation specifies the deterministic evolution of the probability density of $X(t)$.

    Thus, the Fokker-Planck equation is the distribution-level description associated with the underlying SDE.

2. Explain how the probability flow ODE can be used to generate samples from  $p_0\left(\mathbf{x} \right)$.

    The probability flow ODE can be used to generate samples from $p_0(\mathbf{x})$ because it defines a deterministic backward process whose marginal distributions are the same as those of the corresponding diffusion model at every time $t$.

    In diffusion models, the forward process transforms the data distribution $p_0(\mathbf{x})$ into a simple distribution $p_T(\mathbf{x})$, which is usually well approximated by a standard Gaussian distribution. Therefore, generation starts by drawing an initial sample

    $$
    \mathbf{x}(T) \sim p_T(\mathbf{x}) \approx \mathcal{N}(\mathbf{0}, \mathbf{I}).
    $$

    Instead of simulating the backward stochastic differential equation, one can use the probability flow ODE

    $$
    d\mathbf{x}(t)
    =
    \left[
    \mathbf{f}(\mathbf{x},t)
    -
    \frac{1}{2} g^2(t)\nabla_{\mathbf{x}} \log p_t(\mathbf{x})
    \right] dt.
    $$

    This is a deterministic process, since it does not contain a Wiener noise term. Its key property is that it preserves the same marginal probability densities $p_t(\mathbf{x})$ as the diffusion process. Therefore, if we start at time $T$ from a sample of $p_T(\mathbf{x})$ and integrate this ODE backward up to time $0$, the final state has distribution $p_0(\mathbf{x})$.

    Hence, the generation procedure is the following:

    - Draw an initial sample from the simple distribution at the final diffusion time:
    $$
    \mathbf{x}(T) \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).
    $$

    - Integrate the probability flow ODE backward in time from $t=T$ to $t=0$.

    - Obtain the final point $\mathbf{x}(0)$, which is a generated sample from the target data distribution $p_0(\mathbf{x})$.

3. Implement and illustrate the use of this method to generate synthetic images of handwritten digits.

    View all the specific examples in [samplers/](samplers/) directory

4. Indicate how to use this scheme to compute likelihoods. Implement this functionality and illustrate its use.

    The probability flow ODE can also be used to compute likelihoods because it defines a deterministic transformation between the data distribution $p_0(\mathbf{x})$ and a simple distribution $p_T(\mathbf{x})$, which is typically chosen as a standard Gaussian.

    Since the probability flow ODE is deterministic, it can be interpreted as a continuous change of variables. This allows us to track not only the evolution of the state $\mathbf{x}(t)$, but also how its probability density changes over time.

    Let $\mathbf{x}(t)$ follow the probability flow ODE. The evolution of the log-density along this trajectory is given by

    $$
    \frac{d}{dt} \log p_t(\mathbf{x}(t))
    =
    - \nabla_{\mathbf{x}} \cdot \mathbf{f}_{\text{ODE}}(\mathbf{x}(t),t),
    $$

    where $\mathbf{f}_{\text{ODE}}(\mathbf{x},t)$ denotes the drift of the probability flow ODE and $\nabla_{\mathbf{x}} \cdot$ is the divergence operator.

    By integrating this equation from $t=0$ to $t=T$, we obtain

    $$
    \log p_0(\mathbf{x}(0))
    =
    \log p_T(\mathbf{x}(T))
    +
    \int_0^T \nabla_{\mathbf{x}} \cdot \mathbf{f}_{\text{ODE}}(\mathbf{x}(t),t)\,dt.
    $$

    This equation shows that the log-likelihood of a data point $\mathbf{x}(0)$ can be computed by:

    1. Solving the probability flow ODE forward in time from $t=0$ to $t=T$,

    2. Accumulating the change in log-density along the trajectory,
    
    3. Evaluating the final density $\log p_T(\mathbf{x}(T))$, which is known since $p_T$ is a simple distribution (e.g., Gaussian).

## Exercise 2:  Generative AI based on diffusion models: The Ornstein-Uhlenbeck process

In this exercise, the training data is generated by injecting noise using Brownian motion (variance preserving diffusion model).

The stochastic differential equation (SDE) that characterizes this forward diffusion process is

$$
d\mathbf{x}(t) = - \frac{1}{2} \beta(t) \mathbf{x}(t) + \sqrt{\beta(t)} d\mathbf{W}(t)
$$

with $\mathbf{x}(0) \sim p_0\left(\mathbf{x} \right)$

1. What is the distribution of $\mathbf{x}(t)$ assuming that $\mathbf{x}(0) = \mathbf{x}_0$ for arbitrary $t$?

    To find the distribution of $x(t)$ for an arbitrary time $t$, we must solve the SDE given:

    $$
    d\mathbf{x}(t) = -\frac{1}{2}\beta(t)\mathbf{x}(t)dt + \sqrt{\beta(t)}d\mathbf{W}(t)
    $$

    1. **Step 1: Identifying the Integrating Factor**

        This is a linear SDE. To solve it, we use the method of the **integrating factor**. We look for a function $\mu(t)$ that simplifies the left-hand side. Based on the drift coefficient $-\frac{1}{2}\beta(t)$, we define:

        $$
        \mu(t) = e^{\frac{1}{2}\int_0^t \beta(s)ds}
        $$

    2. **Step 2: Applying Itô's Lemma to the Auxiliary Variable**

        Let $\mathbf{y}(t) = \mathbf{x}(t)\mu(t)$ be our auxiliary variable. We apply the **Itô product rule**:

        $$
        d\mathbf{y}(t) = d(\mathbf{x}(t)\mu(t)) = \mu(t)d\mathbf{x}(t) + \mathbf{x}(t)d\mu(t) + d\mathbf{x}(t)d\mu(t)
        $$

        Where:
        * $d\mathbf{x}(t) = -\frac{1}{2}\beta(t)\mathbf{x}(t)dt + \sqrt{\beta(t)}d\mathbf{W}(t)$
        * Since $\mu(t)$ is a deterministic function of $t$, its differential is simply:
            $$
            d\mu(t) = \frac{d}{dt}\left(e^{\frac{1}{2}\int_0^t \beta(s)ds}\right)dt = \frac{1}{2}\beta(t)e^{\frac{1}{2}\int_0^t \beta(s)ds}dt = \frac{1}{2}\beta(t)\mu(t)dt
            $$
        * The second-order term $d\mathbf{x}(t)d\mu(t) = 0$ because $dt \cdot dt = 0$ and $dW \cdot dt = 0$.

    3. **Step 3: Substitution and Cancellation**

        Now, substitute $d\mathbf{x}(t)$ and $d\mu(t)$ into the product rule formula:

        $$
        d\mathbf{y}(t) = \mu(t) \left[ -\frac{1}{2}\beta(t)\mathbf{x}(t)dt + \sqrt{\beta(t)}d\mathbf{W}(t) \right] + \mathbf{x}(t) \left[ \frac{1}{2}\beta(t)\mu(t)dt \right]
        $$

        $$
        d\mathbf{y}(t) = \underbrace{-\frac{1}{2}\beta(t)\mathbf{x}(t)\mu(t)dt + \frac{1}{2}\beta(t)\mathbf{x}(t)\mu(t)dt}_{\text{Cancels to 0}} + \mu(t)\sqrt{\beta(t)}d\mathbf{W}(t)
        $$

        This leaves us with the simplified expression:

        $$
        d(\mathbf{x}(t)e^{\frac{1}{2}\int_0^t \beta(s)ds}) = e^{\frac{1}{2}\int_0^t \beta(s)ds} \sqrt{\beta(t)} d\mathbf{W}(t)
        $$

    4. **Step 4: Integration**

        Integrate both sides from $0$ to $t$:

        $$
        \mathbf{x}(t)e^{\frac{1}{2}\int_0^t \beta(s)ds}-\mathbf{x}(0)e^{0}=\int_0^t e^{\frac{1}{2}\int_0^s \beta(u)du}\sqrt{\beta(s)}dW(s)
        $$

        Multiply by $e^{-\frac{1}{2}\int_0^t \beta(s)ds}$ to solve for $\mathbf{x}(t)$:
        
        $$
        \mathbf{x}(t) = \mathbf{x}_0 e^{-\frac{1}{2}\int_0^t \beta(s)ds} + \int_0^t e^{-\frac{1}{2}\int_s^t \beta(u)du} \sqrt{\beta(s)} dW(s)
        $$

    5. **Step 5: Determining the Gaussian Parameters**
        
        Since the stochastic integral of a deterministic function with respect to a Wiener process is Gaussian with zero mean, $\mathbf{x}(t) \sim \mathcal{N}(\mu_{\mathbf{x}_0}(t), \sigma^2(t)I)$.

        1.  **Mean ($\mu_{\mathbf{x}_0}(t)$):** Taking the expectation $\mathbb{E}[\mathbf{x}(t)|\mathbf{x}_0]$, the stochastic integral vanishes:
            
            $$
            \mu_{\mathbf{x}_0}(t) = \mathbf{x}_0 e^{-\frac{1}{2}\int_0^t \beta(s)ds}
            $$

        2.  **Variance ($\sigma^2(t)$):** Applying **Itô’s Isometry**, the variance is the integral of the squared diffusion term:
            
            $$
            \sigma^2(t) = \int_0^t \left( e^{-\frac{1}{2}\int_s^t \beta(u)du} \sqrt{\beta(s)} \right)^2 ds = \int_0^t e^{-\int_s^t \beta(u)du} \beta(s) ds
            $$
            
            Using the identity $\frac{d}{ds}(e^{-\int_s^t \beta(u)du}) = \beta(s)e^{-\int_s^t \beta(u)du}$, we solve the integral:

            $$
            \sigma^2(t) = \left[ e^{-\int_s^t \beta(u)du} \right]_{s=0}^{s=t} = e^0 - e^{-\int_0^t \beta(s)ds} = 1 - e^{-\int_0^t \beta(s)ds}
            $$

    6. **Final Result:**

        For an arbitrary $t$, the distribution is:

        $$
        p(\mathbf{x}(t)|\mathbf{x}(0) = \mathbf{x}_0) = \mathcal{N}\left(\mathbf{x}(t); \mathbf{x}_0 e^{-\frac{1}{2}\int_0^t \beta(s)ds}, \left(1 - e^{-\int_0^t \beta(s)ds}\right)I\right)
        $$


2. What is the SDE for the time-reversed process?

    We aim to derive the stochastic differential equation (SDE) corresponding to the time-reversed process associated with the forward diffusion:

    $$
    d\mathbf{x}(t) = -\frac{1}{2}\beta(t)\mathbf{x}(t)\,dt + \sqrt{\beta(t)}\,d\mathbf{W}(t)
    $$

    The goal of diffusion models is to reverse this process, transforming samples from a simple distribution into data-like samples.

    1. **Step 1: General form of the forward SDE**

        A general stochastic differential equation can be written as:

        $$
        d\mathbf{x}(t) = \mathbf{f}(\mathbf{x}(t), t)\,dt + g(t)\,d\mathbf{W}(t),
        $$

        where:

        * $\mathbf{f}(\mathbf{x}, t)$ is the drift term,
        * $g(t)$ is the diffusion coefficient.

        Comparing this general form with the given forward process, we identify:

        $$
        \mathbf{f}(\mathbf{x}, t) = -\frac{1}{2}\beta(t)\mathbf{x},
        \qquad
        g(t) = \sqrt{\beta(t)}.
        $$

    2. **Step 2: Reverse-time SDE (general result)**

        From the theory of diffusion models, the reverse-time SDE associated with the forward process is given by:

        $$
        d\mathbf{x}(t) =
        \left[
        \mathbf{f}(\mathbf{x}(t), t)
        - g^2(t)\,\nabla_{\mathbf{x}} \log p_t(\mathbf{x})
        \right] dt
        + g(t)\,d\mathbf{W}(t),
        $$

        where $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$ is the **score function**.

    3. **Step 3: Substitution for the OU process**

        Using the expressions:

        $$
        \mathbf{f}(\mathbf{x}, t) = -\frac{1}{2}\beta(t)\mathbf{x},
        \qquad
        g^2(t) = \beta(t),
        $$

        we substitute into the general formula:

        $$
        d\mathbf{x}(t) =
        \left[
        -\frac{1}{2}\beta(t)\mathbf{x}(t)
        - \beta(t)\nabla_{\mathbf{x}} \log p_t(\mathbf{x})
        \right] dt
        + \sqrt{\beta(t)}\,d\mathbf{W}(t).
        $$

        This is the reverse-time SDE for the Ornstein-Uhlenbeck diffusion process.

    4. **Step 4: Interpretation**

        The reverse SDE consists of three components:

        * The drift term $-\frac{1}{2}\beta(t)\mathbf{x}(t)$, inherited from the forward process.
        * A correction term $-\beta(t)\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$, which steers the process towards regions of high data density.
        * A stochastic noise term $\sqrt{\beta(t)}\,d\mathbf{W}(t)$.

        The key idea is that the score function $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$ encodes the structure of the data distribution.

    5. **Step 5: Practical approximation**

        In practice, the score function is unknown and must be approximated using a neural network:

        $$
        \mathbf{s}_\theta(\mathbf{x}, t) \approx \nabla_{\mathbf{x}} \log p_t(\mathbf{x}).
        $$

        Therefore, the reverse SDE used in practice becomes:

        $$
        d\mathbf{x}(t) =
        \left[
        -\frac{1}{2}\beta(t)\mathbf{x}(t)
        - \beta(t)\mathbf{s}_\theta(\mathbf{x}, t)
        \right] dt
        + \sqrt{\beta(t)}\,d\mathbf{W}(t).
        $$

    6. **Final result**

        The time-reversed SDE associated with the Ornstein-Uhlenbeck forward diffusion process is:

        $$
        d\mathbf{x}(t) =
        \left[
        -\frac{1}{2}\beta(t)\mathbf{x}(t)
        - \beta(t)\nabla_{\mathbf{x}} \log p_t(\mathbf{x})
        \right] dt
        + \sqrt{\beta(t)}\,d\mathbf{W}(t),
        $$

        which, in practice, is approximated using a neural network score model.

3. For the synthesis of new images

    1. In what interval is the reverse process SDE integrated?

        The forward diffusion process evolves from time $t = 0$ to $t = T$, transforming data samples into noise.

        Therefore, the reverse-time SDE must be integrated in the opposite direction, from $t = T$ back to $t = 0$:

        $$
        t \in [T, 0].
        $$

        This corresponds to gradually transforming noise into data-like samples.

    2. From which distribution $\pi\left(\mathbf{x} \right)$ is the initial condition of the reverse SDE sampled?

        The reverse process starts at time $t = T$, where the distribution of $\mathbf{x}(T)$ is close to a simple distribution.

        For the Ornstein-Uhlenbeck process, the forward diffusion converges to a Gaussian distribution as $t$ increases.

        Therefore, the initial condition is sampled from:

        $$
        \pi(\mathbf{x}) = \mathcal{N}(0, \sigma^2(T) I).
        $$

        In practice, this distribution is often approximated by:

        $$
        \pi(\mathbf{x}) = \mathcal{N}(0, I).
        $$

        This allows the reverse SDE to start from pure noise and progressively transform it into realistic samples.

### Exercise 2.1: Training and generation of images using the OU process and different noise schedules

1. Using the linear noise schedule.

    View notebooks for [training](train/train_mnist_ou.ipynb) and [sampling](samplers/sampler_mnist_ou_linear.ipynb)

2. Using the cosine noise schedule.

    View notebooks for [training](train/train_mnist_ou.ipynb) and [sampling](samplers/sampler_mnist_ou_cosine.ipynb)

3. Using a third noise schedule of your choice.

    View notebooks for [training](train/train_mnist_ou.ipynb) and [sampling](samplers/sampler_mnist_ou_sigmoid.ipynb)

## Exercise 3:  Evaluation of the quality of the generated images
1. Describe, and compare the characteristics, advantages, and disadvantages of the following measures of quality for generative AI models for images:
    1. **The negative log-likelihood (NLL).**

        The negative log-likelihood measures how well a generative model approximates the true data distribution. It is defined as the negative logarithm of the probability that the model assigns to the test data.

        **Advantages:**

        - It is a theoretically grounded measure, based on probability.  
        - It allows objective comparison between models.  
        - It is directly related to training in likelihood-based models.  

        **Disadvantages:**

        - It can be difficult to compute for complex models.  
        - As discussed in class, direct likelihood maximization is not always possible and often requires approximations, which may lead to suboptimal results.  
        - It does not reflect the perceptual quality of images well.

    2. **Bits per dimension (BPD).**

        Bits per dimension is a normalized version of the NLL that measures how many bits are required to represent each dimension (e.g., each pixel) of the data.

        **Advantages:**

        - It allows comparison between models with different image sizes.  
        - It has a clear interpretation in terms of information compression.  
        - It is widely used in likelihood-based generative models.  

        **Disadvantages:**

        - Like NLL, it is not aligned with human perception.  
        - A model may achieve a good BPD while generating unrealistic images.  

    3. **Fréchet Inception Distance (FID).**

        FID measures the distance between the feature distributions of real and generated images using a pretrained network (Inception).

        **Advantages:**

        - It evaluates the perceptual quality of images.  
        - It considers both quality and diversity.  
        - It is one of the most widely used metrics in practice.  

        **Disadvantages:**

        - It depends on an external network (Inception), which introduces bias.  
        - It is not a probabilistic measure.  
        - It can be sensitive to dataset size.  

    4. **A third measure of your choice: Inception Score (IS).**

        The Inception Score evaluates generated images using a pretrained classifier, measuring whether images are clear (low conditional entropy) and diverse (high marginal entropy).

        **Advantages:**

        - It is easy to compute.  
        - It captures whether images are recognizable and diverse.  

        **Disadvantages:**

        - It does not directly compare with real data.  
        - It may favor models that generate images that are easy for the classifier.  
        - It is less robust than FID.  

    **Conclusion**

    Likelihood-based metrics (NLL, BPD) are useful from a theoretical perspective but do not reflect visual quality well. On the other hand, metrics such as FID and IS are more aligned with human perception, although they depend on external models and lack a direct probabilistic interpretation. Therefore, in practice, multiple metrics are usually used in a complementary way.

2. Compare using BPD the different diffusion models (Brownian motion, Ornstein-Uhlenbeck), noising schedules (linear, cosine, etc.), and sampling strategies (SDE, ODE) implemented.

    View all the notebooks in [measures/](measures/)

## References:

1. Anderson, B. D. O. (1982). Reverse-time diffusion equation models. Stochastic Processes and their Applications.

2. Alberto Suárez (2025). Lecture notes on diffusion models, Aprendizaje Automático III, Universidad Autónoma de Madrid.

2. Hyvärinen, A. (2005). Estimation of non-normalized statistical models by score matching. Journal of Machine Learning Research.

3. Sohl-Dickstein, J., Weiss, E., Maheswaranathan, N., \& Ganguli, S. (2015). Deep unsupervised learning using nonequilibrium thermodynamics. ICML.

4. Song, Y., Sohl-Dickstein, J., Kingma, D. P., Kumar, A., Ermon, S., \& Poole, B. (2021). Score-based generative modeling through stochastic differential equations. ICLR.

5. Ho, J., Jain, A., \& Abbeel, P. (2020). Denoising diffusion probabilistic models. NeurIPS.

For the complete references and additional details, see the report: [DiffusionModels_Report](DiffusionModels_Report.pdf).
